In [8]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Ratings file
ratings = pd.read_csv("ml-100k/u.data", sep="\t", names=["userId", "itemId", "rating", "timestamp"])
ratings = ratings[["userId", "itemId", "rating"]]

# Movie info file
movies = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1", header=None, usecols=[0,1], names=["itemId", "title"])

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)


Ratings: (100000, 3)
Movies: (1682, 2)


In [9]:
test = ratings.groupby("userId").sample(n=2, random_state=42)
train = ratings.drop(test.index)

In [10]:

test = ratings.groupby("userId").sample(n=2, random_state=42)
train = ratings.drop(test.index)

# Create user-item matrix (training only)
user_item = train.pivot_table(index="userId", columns="itemId", values="rating").fillna(0)

#Mean-center ratings for better similarity
user_item_centered = user_item.sub(user_item.mean(axis=1), axis=0).fillna(0)

#Compute user-user cosine similarity
user_sim = cosine_similarity(user_item_centered)
np.fill_diagonal(user_sim, 0)
user_sim = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)



In [11]:
#Recommendation function
def recommend_movies(user_id, k=30, top_n=10):
    if user_id not in user_sim.index:
        return []
    sims = user_sim[user_id].sort_values(ascending=False)[:k]
    weighted_ratings = user_item.loc[sims.index].T.dot(sims) / sims.sum()
    rated_items = user_item.loc[user_id] > 0
    weighted_ratings[rated_items] = 0
    top_items = weighted_ratings.sort_values(ascending=False)[:top_n].index
    # Map IDs to movie titles
    recommended_titles = movies[movies.itemId.isin(top_items)]["title"].tolist()
    return recommended_titles





In [12]:
#Precision@K function
def precision_at_k(recommended, actual, k):
    rec_k = recommended[:k]
    hits = len(set(rec_k) & set(actual))
    return hits / k if k > 0 else 0

# Evaluate for all users
k = 5
precisions = []

for user in test["userId"].unique():
    actual_ids = test[test.userId == user]["itemId"].tolist()
    actual_titles = movies[movies.itemId.isin(actual_ids)]["title"].tolist()
    recommended_titles = recommend_movies(user, k=40, top_n=k)
    if len(recommended_titles) > 0:
        precisions.append(precision_at_k(recommended_titles, actual_titles, k))

avg_precision = np.mean(precisions)
print(f"\nAverage Precision {k}: {avg_precision:.4f}")

# 9️⃣ Example recommendations for one user
user_id = 10
print(f"\nTop {k} movie recommendations for User {user_id}:")
for i, movie in enumerate(recommend_movies(user_id, k=40, top_n=k), 1):
    print(f"{i}. {movie}")


Average Precision 5: 0.0776

Top 5 movie recommendations for User 10:
1. Blade Runner (1982)
2. Empire Strikes Back, The (1980)
3. Schindler's List (1993)
4. To Kill a Mockingbird (1962)
5. Annie Hall (1977)
